# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/earlykisses/flyrank_mlinternship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

# ML-04 — Search Intelligence Data Contract

## 1. Unit of Analysis + Time Window

### Unit of Analysis

One row represents the daily performance of a single content item for a single client on one report date.

### Tables Used

- fact_content_daily_performance (primary)
- dim_content (optional for content metadata)
- dim_clients (optional for client history and availability checks)

### Time Window

This notebook uses a mid-panel month (2026-03) for development, as recommended in the FlyRank data guide. The final month (June 2026) is treated as a sealed test period and is not used for developing label logic.

### Prediction Target

The objective is to rank content items according to their refresh opportunity so that editors can prioritize which pages should be updated first.

### Deliberately Excluded

The following are excluded from model features:

- client_id (context only)
- content_id (context only)
- any future-derived or label-derived information

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [9]:
!git clone https://github.com/earlykisses/flyrank_mlinternship.git

Cloning into 'flyrank_mlinternship'...
remote: Enumerating objects: 127, done.
remote: Counting objects: 100% (127/127), done.
remote: Compressing objects: 100% (83/83), done.
remote: Total 127 (delta 41), reused 93 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (127/127), 1.85 MiB | 5.25 MiB/s, done.
Resolving deltas: 100% (41/41), done.


In [10]:
%cd flyrank_mlinternship


/content/flyrank_mlinternship/flyrank_mlinternship


In [11]:
!pip -q install duckdb pandas pyarrow datasets huggingface_hub

In [12]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("Token loaded successfully!" if HF_TOKEN else "Token not found.")

Token loaded successfully!


In [17]:
from huggingface_hub import login

login(token=HF_TOKEN)

In [18]:
import duckdb

con = duckdb.connect()

In [19]:
con.execute("""
INSTALL httpfs;
LOAD httpfs;
""")

In [21]:
from huggingface_hub import list_repo_files

files = list_repo_files(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
)

for f in files:
    print(f)

.gitattributes
README.md
dim_clients.parquet
dim_content.parquet
fact_content_daily_performance/month=2025-01/data_0.parquet
fact_content_daily_performance/month=2025-02/data_0.parquet
fact_content_daily_performance/month=2025-03/data_0.parquet
fact_content_daily_performance/month=2025-04/data_0.parquet
fact_content_daily_performance/month=2025-05/data_0.parquet
fact_content_daily_performance/month=2025-06/data_0.parquet
fact_content_daily_performance/month=2025-07/data_0.parquet
fact_content_daily_performance/month=2025-08/data_0.parquet
fact_content_daily_performance/month=2025-09/data_0.parquet
fact_content_daily_performance/month=2025-10/data_0.parquet
fact_content_daily_performance/month=2025-11/data_0.parquet
fact_content_daily_performance/month=2025-12/data_0.parquet
fact_content_daily_performance/month=2026-01/data_0.parquet
fact_content_daily_performance/month=2026-02/data_0.parquet
fact_content_daily_performance/month=2026-03/data_0.parquet
fact_content_daily_performance/mont

In [25]:
from huggingface_hub import hf_hub_download

dim_clients = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="dim_clients.parquet",
    token=HF_TOKEN,
)

dim_content = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="dim_content.parquet",
    token=HF_TOKEN,
)

march_data = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    token=HF_TOKEN,
)

print(dim_clients)
print(dim_content)
print(march_data)

dim_content.parquet: reconstructing file:   0%|          |  0.00B / 19.6MB            

dim_content.parquet: downloading bytes:           |  0.00B            

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/dim_clients.parquet
/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/dim_content.parquet
/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-03/data_0.parquet


In [26]:
con.execute(f"""
SELECT *
FROM read_parquet('{dim_clients}')
LIMIT 5;
""").df()

,client_hash_id,is_active,has_gsc_access,has_ga4_access,access_profile,client_created_date,client_updated_date,gsc_data_start,ga4_data_start
0,client_04660893ae39614a,True,True,True,gsc_and_ga4,2026-04-15,2026-06-27,NaT,2026-05-22
1,client_05475c07ed21a83a,True,False,False,no_search_or_analytics_access,2026-04-01,2026-06-27,NaT,NaT
2,client_06d356715a8ff3b6,True,True,True,gsc_and_ga4,2026-03-23,2026-07-05,2026-04-10,2026-04-06
3,client_0797ff3a1fc9a6a5,True,False,False,no_search_or_analytics_access,2025-05-26,2026-06-27,2025-11-05,NaT
4,client_08a6a72ff48e62c0,True,True,False,gsc_only,2025-05-26,2026-06-27,2025-09-24,NaT


In [27]:
con.execute(f"""
SELECT *
FROM read_parquet('{dim_content}')
LIMIT 5;
""").df()

,client_hash_id,content_hash_id,keyword_hash_id,url_hash_id,keyword_char_count,keyword_token_count,url_char_count,content_created_date,content_updated_date,content_type,...,category_count,keyword_created_date,provider_used,model_used,char_count,word_count,last_optimized_date,optimization_eligible_date,is_published,is_deleted
0,client_04660893ae39614a,content_004de9653278b5a4,keyword_e754999ab88dd9f2,url_d6091f18cf628794,22,4,108,2026-05-30,2026-07-01,keyword article,...,3,2026-05-12,gemini-generate-content,gemini-3-flash-preview,15682,2555,NaT,NaT,True,False
1,client_04660893ae39614a,content_00dc5efae381b2ab,keyword_4329d7aede8e208b,url_3a66d2f2e36823ca,31,6,95,2026-06-12,2026-07-01,keyword article,...,4,2026-06-01,gemini-generate-content,gemini-3-flash-preview,15438,2430,NaT,NaT,True,False
2,client_04660893ae39614a,content_01410f2556c327ac,keyword_9b08047d3d2a0406,url_809eda7a7e20b3b2,22,5,82,2026-05-09,2026-07-01,keyword article,...,4,2026-05-06,gemini-generate-content,gemini-3-flash-preview,16576,2645,NaT,NaT,True,False
3,client_04660893ae39614a,content_019f27f634053ca7,keyword_e7cec7ab1804c1c2,url_5fb42bafc4399861,14,3,92,2026-06-15,2026-06-15,keyword article,...,4,2026-06-01,gemini-generate-content,gemini-3-flash-preview,15457,2522,NaT,NaT,True,False
4,client_04660893ae39614a,content_01efa71faea45dcc,keyword_56b0062a1d8b7524,url_ece0abc3e5fb75f9,24,6,98,2026-05-21,2026-06-01,keyword article,...,4,2026-05-12,gemini-generate-content,gemini-3-flash-preview,15776,2552,NaT,NaT,True,False


In [28]:
march_df = con.execute(f"""
SELECT *
FROM read_parquet('{march_data}')
LIMIT 5;
""").df()

march_df

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


In [29]:
schema = con.execute(f"""
DESCRIBE
SELECT *
FROM read_parquet('{march_data}');
""").df()

schema


,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


In [30]:
print(schema)

                 column_name column_type null   key default extra
0                report_date        DATE  YES  None    None  None
1             client_hash_id     VARCHAR  YES  None    None  None
2            content_hash_id     VARCHAR  YES  None    None  None
3             client_has_gsc     BOOLEAN  YES  None    None  None
4             client_has_ga4     BOOLEAN  YES  None    None  None
5         gsc_data_available     BOOLEAN  YES  None    None  None
6         ga4_data_available     BOOLEAN  YES  None    None  None
7            gsc_impressions      BIGINT  YES  None    None  None
8                 gsc_clicks      BIGINT  YES  None    None  None
9           gsc_sum_position      BIGINT  YES  None    None  None
10          gsc_avg_position      DOUBLE  YES  None    None  None
11             ga4_pageviews      BIGINT  YES  None    None  None
12              ga4_sessions      BIGINT  YES  None    None  None
13                 ga4_users      BIGINT  YES  None    None  None
14      ga

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [31]:
con.execute(f"""
SELECT COUNT(*) AS total_rows
FROM read_parquet('{march_data}');
""").df()

,total_rows
0,9841378


In [32]:
con.execute(f"""
SELECT
MIN(report_date) AS start_date,
MAX(report_date) AS end_date
FROM read_parquet('{march_data}');
""").df()

,start_date,end_date
0,2026-03-01,2026-03-31


In [34]:
con.execute(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS duplicate_rows
FROM read_parquet('{march_data}')
GROUP BY
    report_date,
    client_hash_id,
    content_hash_id
HAVING COUNT(*) > 1
LIMIT 10;
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,duplicate_rows


In [35]:
con.execute(f"""
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date
FROM read_parquet('{march_data}');
""").df()

,total_rows,start_date,end_date
0,9841378,2026-03-01,2026-03-31


In [36]:
con.execute(f"""
SELECT
    COUNT(*) AS rows_with_ga4
FROM read_parquet('{march_data}')
WHERE ga4_data_available IS TRUE;
""").df()

,rows_with_ga4
0,413966


In [37]:
con.execute(f"""
SELECT
    COUNT(*) AS rows_with_gsc
FROM read_parquet('{march_data}')
WHERE gsc_data_available IS TRUE;
""").df()

,rows_with_gsc
0,3611061


In [38]:
feature_df = con.execute(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    ga4_pageviews,
    ga4_sessions
FROM read_parquet('{march_data}')
WHERE
    ga4_data_available IS TRUE
LIMIT 20;
""").df()

feature_df.head()

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,ga4_sessions
0,2026-03-01,client_65de48885f4ef01b,content_09be8cc7fcb222af,0,0,NaN,1,1
1,2026-03-01,client_65de48885f4ef01b,content_851afac9fe13612e,0,0,NaN,1,1
2,2026-03-01,client_65de48885f4ef01b,content_cee6c6fc8c51af14,0,0,NaN,1,1
3,2026-03-01,client_65de48885f4ef01b,content_5e120e972f11f833,0,0,NaN,1,1
4,2026-03-01,client_65de48885f4ef01b,content_16a7291bb6ecaebe,0,0,NaN,1,1


### Selected Features

| Feature | Available at decision time because... |
|----------|----------------------------------------|
| gsc_impressions | Historical search impressions are already known before deciding whether to refresh a page. |
| gsc_clicks | Historical search clicks are available before the refresh decision. |
| gsc_avg_position | Historical average ranking position is known from Search Console before making the decision. |
| ga4_pageviews | Historical page views have already been collected before the prediction date. |
| ga4_sessions | Historical user sessions are available before deciding which pages to refresh. |

## Data Contract

### Unit of Analysis

One row represents the daily performance of one content item for one client on one report date.

### Table

Primary table:
- fact_content_daily_performance

Supporting tables:
- dim_clients
- dim_content

### Time Window

Development is performed on the March 2026 partition (`month=2026-03`), following the internship guidance to avoid using the final month.

### Prediction

Rank content items according to their refresh opportunity so editors can prioritize updates.

### Excluded

- client_hash_id (identifier only)
- content_hash_id (identifier only)
- any future or label-derived information

## Data Limits

- Client histories begin at different dates.
- GA4 metrics are unavailable when `ga4_data_available` is FALSE.
- Search Console metrics are unavailable when `gsc_data_available` is FALSE.
- Only one monthly partition is analyzed in this notebook.
- External factors such as algorithm updates and competitor actions are not represented in the warehouse.

## 2. Fields

### Features

- clicks
- impressions
- gsc_avg_position
- ctr
- engagement_rate (or another historical GA4 metric if available)

These are historical signals that are available before making a refresh decision.

### Label / Proxy

Refresh opportunity score (or the label/proxy defined by the notebook).

The label is never used as an input feature.

### Context

- client_id
- content_id
- report_date

These identify records or support grouping and analysis but are not model inputs.

### Excluded

- client_id (identifier only)
- content_id (identifier only)
- any label-derived or future-derived fields

Reason:
They either identify entities instead of content quality or introduce data leakage.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [40]:
# Row count and time window
con.execute(f"""
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date
FROM read_parquet('{march_data}');
""").df()

,total_rows,start_date,end_date
0,9841378,2026-03-01,2026-03-31


In [41]:
# Verify grain
con.execute(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS duplicate_rows
FROM read_parquet('{march_data}')
GROUP BY
    report_date,
    client_hash_id,
    content_hash_id
HAVING COUNT(*) > 1;
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,duplicate_rows


In [42]:
# Missing values
con.execute(f"""
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN ga4_data_available IS FALSE THEN 1 ELSE 0 END) AS missing_ga4,
    SUM(CASE WHEN gsc_data_available IS FALSE THEN 1 ELSE 0 END) AS missing_gsc
FROM read_parquet('{march_data}');
""").df()

,total_rows,missing_ga4,missing_gsc
0,9841378,6408671.0,6230317.0


In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## Data Limits

- This notebook uses only the March 2026 partition, so observations may not generalize to all months.
- Some rows do not contain GA4 metrics because `ga4_data_available` is FALSE.
- Some rows do not contain Search Console metrics because `gsc_data_available` is FALSE.
- Client histories begin at different times, resulting in unequal historical coverage.
- External events such as Google algorithm updates or marketing campaigns are not represented in the warehouse.

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


- [x] Every section above is filled
- [x] Notebook runs top to bottom
- [x] No client names or URLs
- [x] Claims use careful wording
- [x] Committed to work/notebooks